In [1]:
# !pip install langchain-ollama duckduckgo-search -q

import time
from langchain_ollama import ChatOllama
from langchain.tools import tool
from duckduckgo_search import DDGS
from langchain_core.messages import HumanMessage

print("✅ Библиотеки установлены\n")

✅ Библиотеки установлены



In [2]:
# ====================== LLM (Облачная модель) ======================
llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    temperature=0.65,
    base_url="http://localhost:11434",
    # num_ctx=8192
)

print("✅ Модель gpt-oss:20b-cloud загружена (облачная)\n")

✅ Модель gpt-oss:20b-cloud загружена (облачная)



In [3]:
# ====================== Инструмент поиска ======================
@tool
def web_search(query: str) -> str:
    """Поиск актуальной информации в интернете"""
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=7)
            formatted = "\n\n".join([
                f"Заголовок: {r.get('title', '')}\n"
                f"Сниппет: {r.get('body', '')[:300]}...\n"
                f"Ссылка: {r.get('href', '')}"
                for r in results
            ])
            return formatted if formatted else "Результатов не найдено."
    except Exception as e:
        return f"Ошибка поиска: {str(e)}"

In [1]:
# ====================== Тема ======================
topic = "Сравнение открытых (open-source) и закрытых больших языковых моделей в 2026 году"

print("🔍 Исследователь начинает работу...\n")

# Researcher
research_prompt = f"""
Ты — эксперт-исследователь ИИ. Проведи глубокий анализ по теме: {topic}.

Используй инструмент поиска, чтобы получить свежую информацию.
Особое внимание удели:
- Поддержке русского языка
- Актуальным моделям 2026 года (Llama, Qwen, DeepSeek, GPT, Claude, Gemini и др.)
- Преимуществам и недостаткам open-source vs closed-source
"""

research_chain = llm.bind_tools([web_search])
research_response = research_chain.invoke([HumanMessage(content=research_prompt)])

# Выполняем инструмент, если был вызов
if hasattr(research_response, 'tool_calls') and research_response.tool_calls:
    tool_call = research_response.tool_calls[0]
    if tool_call['name'] == 'web_search':
        search_result = web_search.invoke(tool_call['args']['query'])
        research_final = llm.invoke(f"""
            {research_prompt}
            
            Результаты поиска:
            {search_result}
            
            Напиши подробный исследовательский отчёт на русском языке.
        """)
    else:
        research_final = research_response
else:
    research_final = research_response

print("✅ Исследование завершено\n")
print("✍️ Писатель пишет финальную статью...\n")

# Writer
writer_prompt = f"""
На основе исследования ниже напиши **увлекательную и образовательную статью** для лекции по AI Agents.

{research_final.content if hasattr(research_final, 'content') else research_final}

Требования:
- Живой, доступный язык
- Хорошая структура (заголовки, списки, примеры)
- Только чистый текст статьи на русском
- Подходит для студентов и разработчиков
"""

final_article = llm.invoke(writer_prompt).content

# ====================== Вывод ======================
print("\n" + "="*90)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ — СТАТЬЯ ДЛЯ ЛЕКЦИИ")
print("="*90)
print(final_article)

with open("09_2_agent writer.md", "w", encoding="utf-8") as f:
    f.write(final_article)

print("\n✅ Готово! Результат сохранён в 09_2_agent writer.md")

✅ Библиотеки установлены

✅ Модель gpt-oss:20b-cloud загружена (облачная)

🔍 Исследователь начинает работу...



/tmp/ipykernel_2642995/4194709189.py:31: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


✅ Исследование завершено

✍️ Писатель пишет финальную статью...


ФИНАЛЬНЫЙ РЕЗУЛЬТАТ — СТАТЬЯ ДЛЯ ЛЕКЦИИ
**AI‑агенты в 2026 году: открытые и закрытые языковые модели в действии**  
*Образовательная статья для лекции*  

---

### 1. Что такое AI‑агент и почему LLM‑ы важны  

AI‑агент – это программный «ум», который может воспринимать окружение, принимать решения и действовать в соответствии с целями. Ключевым компонентом большинства современных агентов является **Large Language Model (LLM)** – модель, способная генерировать, интерпретировать и обобщать естественный язык.  

- **Вход**: запросы пользователя, данные датчиков, контекст.  
- **Обработка**: LLM прогнозирует следующее слово и формирует ответ.  
- **Выход**: текст, действия в приложении, команды устройствам.  

Начиная с 2024 года, LLM‑ы стали «ядром» почти всех AI‑агентов: чат‑боты, виртуальные ассистенты, системы поддержки, автоматизированные рабочие процессы и даже сложные роботы.  

---

### 2. Состояние рынка LLM в 2026 